# K4TQ — climbing, diving and manoeuvring: K4TQ-A2 against K4TQ-F2

| design | what it is |
|---|---|
| **K4TQ-A2** | Cathy (A1: NACA 2412 on both wings) fully autonomous: Jetson Orin NX, stereo + downward perception |
| **K4TQ-F2** | Freddy (F1: front NACA 4415, rear NACA 2415) fully autonomous, the same avionics |

Everything comes from `designs/k4tq_envelope.py` on the Stage 1 models (`k4tq_sizing`: drive maps, tandem lattice,
each wing's NeuralFoil polar, the stall rule, the transition corridor) and the CAD masses and inertia. Labels:
ESTIMATE (strip theory and rigid-body formulas: roll, pitch, yaw), CALC (drive maps, polars, corridor), MOCK where the
BEMT ran coarsened under `DEBUG = True`. No CFD and no flight test behind any number here.

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import k4tq, k4tq_envelope as ke
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)
DESIGNS = ["A2", "F2"]          # any k4tq variant: "A1", "F1", "E", "C"
print(ke.__doc__)

## 1. The study (about two minutes under the mocks)

In [ ]:
res = ke.study(DESIGNS)
ke.summary(res).T

## 2. Figures: climb on the wings, glide, turns, powered descent

In [ ]:
fig = ke.figure(res, Path(k4tq.__file__).parent / "data" / "k4tq" / "figures" / "k4tq_envelope.png")
Image(str(fig))

## 3. Climb

In [ ]:
for v, r in res.items():
    print(v, r["vertical climb"]); print(v, r["mission climb"])
display(pd.concat({v: r["climb"] for v, r in res.items()}))

## 4. Descent: glide, the steepest dive, the powered descent, the vertical descent

In [ ]:
for v, r in res.items():
    print(v, r["steepest dive"]); print(v, r["vertical descent"])
    print(v, "steepest balanced powered path [deg] by speed:", ke.steepest_powered(r["powered descent"]).to_dict())
display(pd.concat({v: r["glide"] for v, r in res.items()}))

In [ ]:
display(pd.concat({v: r["powered descent"] for v, r in res.items()}))

## 5. Manoeuvre: turns, roll, pitch, yaw, the hover

In [ ]:
for v, r in res.items():
    for k in ("roll", "pitch", "yaw", "hover agility"):
        print(v, k, r[k])
display(pd.concat({v: r["turns"] for v, r in res.items()}))

## Reading it

- **Climb.** On the wings F2 climbs better (≈9.6 against 8.8 m/s at 80 % thrust): the NACA 4415 front wing carries
  the climb's higher cl with less profile drag. Vertically both are the same (the props do the work, ≈11.7 m/s at
  full thrust). Climbing 120 m on the wings costs about a third of the energy of climbing vertically (≈5 against ≈15 Wh):
  leave the roof in the hover, transition low, climb on the wings.
- **Dive / descent.** Motors off, both glide at ≈10:1 (least sink ≈2 m/s). Clean, with zero thrust, neither can dive
  steeper than ≈8° at V_ne — the airframe is clean, so a steep fast dive needs a brake. The brake is the tilt: at
  12 m/s, wings tilted ≈55° and the props holding the weight, the path goes to −20° and beyond (≈4 m/s sink); at
  18 m/s to ≈−14°; above the stall speed, wing-borne, only ≈−4…−5° balances with positive thrust (steeper would need
  windmilling props — not modelled). A vertical descent must stay under ≈2.3 m/s (0.3 vᵢ) for the vortex ring;
  descending 120 m on the wings in a spiral takes about as long and almost no energy.
- **Manoeuvre.** F2 turns tighter (≈41 against 38 deg/s sustained, radius ≈37 against 41 m) on its higher CL max;
  A2 rolls slightly faster (≈259 against 248 deg/s, the same order: ≈0.3 s to 60° bank). In the hover both have the
  same authority (roll/pitch ≈2700/1600 deg/s² from the thrust margin, yaw ≈470 deg/s² from the wings' differential
  tilt — far more yaw than a quadcopter's reaction torque gives).
- **Verdict.** F2 is the better flyer in the climb and the turn, at no measurable cost in the hover or the roll —
  consistent with the comparison notebook (`exp/K4TQ_compare.ipynb`), where Freddy also fits the Ø16 spar.

Caveats: the corridor's electrical power at low thrust in the descent comes from the coarsened mock drive maps and is
pessimistic; roll/pitch/yaw are strip-theory estimates without the actuators' rate limits (a servo at ≈0.1 s/60° will
cap the roll's onset). Both belong in the Stage 3 checks.